# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [ ]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [ ]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [ ]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]
target = "taxa_conversao_pct"


In [ ]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x = "taxa_abandono_carrinho_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
)
fig.show()

In [ ]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x_2 = "profundidade_scroll_pct"

if variavel_x_2 not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x_2,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
)
fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

## Motivo da escolha das variáveis.

Para escolher as variáveis, analisei a relação entre as variáveis de entrada e a taxa_conversao_pct. Dessa forma, a taxa de conversão foi usada como variável resposta, enquanto taxa_abandono_carrinho_pct e profundidade_scroll_pct foram escolhidas para a análise de sensibilidade. Além disso, escolhi taxa_abandono_carrinho_pct porque ela apresentou a relação mais forte com a conversão, com correlação negativa de aproximadamente -0.643. Isso indica que maiores taxas de abandono tendem a estar associadas a menores taxas de conversão. Podemos observar no gráfico de dispersão também, em que os pontos apresentam uma tendência decrescente à medida que a variável taxa_abandono_carrinho_pct aumenta.

Também escolhi profundidade_scroll_pct porque ela teve a segunda relação mais forte com a conversão, com correlação positiva de aproximadamente 0.485. Isso mostra que maiores níveis de profundidade de scroll tendem a estar associados a maiores taxas de conversão.

## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [ ]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.317
1,RMSE,0.402


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O MAE indica o erro médio absoluto do modelo, ou seja, mostra em média o quanto as previsões ficaram distantes dos valores reais da taxa de conversão. Já o RMSE também mede o erro das previsões, mas dá mais peso para erros maiores, ajudando a perceber se o modelo está fazendo desvios mais relevantes. No modelo, o MAE foi de aproximadamente 0.317 e o RMSE foi de aproximadamente 0.402. Isso significa que, em média, as previsões ficam cerca de 0.32% de distância dos valores reais da taxa de conversão.

Considerando que a taxa de conversão média está próxima de 5.87%, esse erro ainda parece aceitável para uma análise exploratória. Mesmo assim, ele deve ser considerado na interpretação dos resultados, já que o modelo foi ajustado usando só duas variáveis de entrada e ainda existe uma diferença entre os valores previstos e os valores reais observados.

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [ ]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032},
 5.868747841831929)

In [ ]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.598,-4.612,-0.461
1,profundidade_scroll_pct,62.449,68.694,5.869,6.015,2.499,0.250


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

Comparando os índices de sensibilidade, a variável que mais impactou a taxa de conversão foi taxa_abandono_carrinho_pct. Quando a taxa de abandono do carrinho aumentou 10%, a conversão prevista caiu de aproximadamente 5.869 para 5.598. Isso gerou uma variação de -4.612% na saída e um índice de sensibilidade de -0.461. Ou seja, quando o abandono aumenta, a conversão tende a cair de forma relevante. Já no caso da profundidade_scroll_pct, um aumento de 10% fez a conversão prevista subir de aproximadamente 5.869 para 6.015. A variação foi de aproximadamente 2.5%, com índice de sensibilidade de 0.250.

Com isso, dá para concluir que a taxa de abandono do carrinho tem um maior impacto sobre a conversão, pois o seu índice de sensibilidade é maior em valor absoluto. Isso implica que abaixar o abandono do carrinho provavelmente teria mais efeito na conversão do que só aumentar a profundidade de scroll.

## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

Com base na análise, eu recomendaria priorizar melhorias para reduzir a taxa_abandono_carrinho_pct, já que essa foi a variável que teve maior impacto sobre a conversão. Na tabela de sensibilidade, quando o abandono do carrinho aumentou 10%, a conversão prevista caiu de 5.869 para 5.598, com variação de -4.612% e índice de sensibilidade de -0.461. Em comparação, a profundidade_scroll_pct também teve efeito positivo, mas menor, com aumento de 10%, a conversão subiu para 6.015, com índice de sensibilidade de 0.250.

Dito isso, temos indícios que faria mais sentido atuar primeiro no fluxo de checkout, tentando reduzir pontos de atrito que levam o usuário a abandonar o carrinho. Algumas ações possíveis seriam simplificar etapas, deixar frete e taxas mais claros antes da finalização e tornar o processo de compra mais rápido e direto.

Uma limitação dessa análise é que ela trabalha com um modelo simplificado, usando apenas duas variáveis e assumindo uma relação linear com a taxa de conversão. Na prática, a conversão também pode depender de outros fatores, como preço, promoções, perfil dos usuários, sazonalidade e experiência geral no aplicativo. Além disso, os resultados indicam associação entre as variáveis, mas não provam necessariamente uma relação de causa e efeito.

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [ ]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])

previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.855
std,0.331
min,4.879
10%,5.423
25%,5.635
50%,5.856
75%,6.075
90%,6.285
max,6.956


In [ ]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A simulação de Monte Carlo mostra que, considerando variações nas variáveis de entrada, a taxa de conversão prevista fica concentrada em torno de 5.855%, com mediana de 5.856%. A maior parte dos cenários simulados ficou entre aproximadamente 5.423% e 6.285%, considerando os percentis de 10% e 90%.

Isso provalvemnte mostra que a recomendação de atuar sobre o abandono do carrinho tem um risco não tão alto, mesmo com incerteza nos dados, as previsões continuam próximas da taxa média de conversão. Só que como existem cenários em que a conversão pode cair para valores próximos de 4.879%, ainda é importante acompanhar os resultados depois da mudança e validar se a ação realmente melhora a conversão na prática.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.